# MLA Model Chat Demo
Run the MLA latent-cache model (or baseline GQA) locally with a chat template and a simple prompt.

In [ ]:
import sys, os
# Point at local methods/ so transmla imports resolve without Modal
repo_root = os.path.abspath('.')  # adjust if running from elsewhere
for p in [repo_root, os.path.join(repo_root, 'methods')]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PAIR = "llama2"   # "llama2" or "qwen25"
USE_MLA    = True       # False → load base GQA model instead
MAX_NEW_TOKENS = 200
DTYPE = torch.float16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATHS = {
    "llama2": {
        "base": "meta-llama/Llama-2-7b-chat-hf",
        "mla":  "botsxc/llama2-7b-chat-mla-2048-8",
    },
    "qwen25": {
        "base": "Qwen/Qwen2.5-7B",
        "mla":  "botsxc/qwen2.5-7b-instruct-mla-256",
    },
}

model_path = MODEL_PATHS[MODEL_PAIR]["mla" if USE_MLA else "base"]
print(f"Loading: {model_path}  (MLA={USE_MLA}, device={DEVICE})")

In [ ]:
# ── Load model & tokenizer ────────────────────────────────────────────────────
if USE_MLA:
    from transmla.transformers.llama.configuration_llamamla import LlamaMLAConfig
    from transmla.transformers.llama.modeling_llamamla_latent import LlamaMLALatentForCausalLM

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    config    = LlamaMLAConfig.from_pretrained(model_path)
    model     = LlamaMLALatentForCausalLM.from_pretrained(
        model_path, config=config, torch_dtype=DTYPE, device_map=DEVICE
    )
else:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=DTYPE, device_map=DEVICE
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("Model loaded.")

In [ ]:
# ── Chat template helper ──────────────────────────────────────────────────────
def apply_chat_template(tokenizer, messages: list[dict]) -> str:
    """
    Use the tokenizer's built-in chat template when available.
    Falls back to a simple Llama-2-style [INST] wrapper.
    """
    if hasattr(tokenizer, 'chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    # Llama-2 chat fallback
    system = next((m['content'] for m in messages if m['role'] == 'system'), '')
    user   = next((m['content'] for m in messages if m['role'] == 'user'), '')
    if system:
        return f"[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{user} [/INST]"
    return f"[INST] {user} [/INST]"

In [ ]:
# ── Generate function (baseline or MLA latent) ────────────────────────────────
def generate(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> tuple[str, dict]:
    import time
    inputs    = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_len = inputs["input_ids"].shape[1]

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
        torch.cuda.synchronize(DEVICE)

    t0 = time.perf_counter()

    if USE_MLA:
        from transmla.transformers.mla_latent import MLALatentCache
        cache = MLALatentCache()
        with torch.no_grad():
            prefill_out = model(**inputs, past_key_values=cache, use_cache=True)
    else:
        with torch.no_grad():
            prefill_out = model(**inputs, use_cache=True)

    past_kv = prefill_out.past_key_values
    if DEVICE == "cuda":
        torch.cuda.synchronize(DEVICE)
    ttft = time.perf_counter() - t0

    next_token     = prefill_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated_ids  = []
    t_decode_start = time.perf_counter()

    for _ in range(max_new_tokens):
        if next_token.item() == tokenizer.eos_token_id:
            break
        generated_ids.append(next_token.item())
        with torch.no_grad():
            step_out = model(input_ids=next_token, past_key_values=past_kv, use_cache=True)
        past_kv    = step_out.past_key_values
        next_token = step_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    if DEVICE == "cuda":
        torch.cuda.synchronize(DEVICE)
    t_end = time.perf_counter()

    n_gen       = len(generated_ids)
    decode_time = t_end - t_decode_start

    if USE_MLA:
        kv_mb = past_kv.get_cache_bytes() / 1e6
    else:
        from benchmark.runner import _cache_to_tuple
        kv_tuples = _cache_to_tuple(past_kv)
        kv_mb = sum(k.element_size()*k.numel() + v.element_size()*v.numel() for k,v in kv_tuples) / 1e6

    metrics = {
        "input_tokens":        input_len,
        "tokens_generated":    n_gen,
        "ttft_ms":             ttft * 1000,
        "throughput_tps":      n_gen / decode_time if decode_time > 0 else 0.0,
        "per_token_latency_ms":decode_time / n_gen * 1000 if n_gen > 0 else 0.0,
        "kv_cache_mb":         kv_mb,
    }
    if DEVICE == "cuda":
        metrics["peak_memory_gb"] = torch.cuda.max_memory_allocated(DEVICE) / 1e9

    return tokenizer.decode(generated_ids, skip_special_tokens=True), metrics

In [ ]:
# ── Define your conversation ──────────────────────────────────────────────────
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain what multi-head latent attention (MLA) is in two sentences."},
]

prompt = apply_chat_template(tokenizer, messages)
print("=== Formatted prompt ===")
print(repr(prompt[:300]))

In [ ]:
# ── Run generation ────────────────────────────────────────────────────────────
response, metrics = generate(prompt)

print("=== Response ===")
print(response)
print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"  {k:<25} {v:.3f}")

In [ ]:
# ── Optional: try a different prompt without reloading the model ──────────────
messages2 = [
    {"role": "user", "content": "What is the capital of France?"},
]
prompt2 = apply_chat_template(tokenizer, messages2)
response2, metrics2 = generate(prompt2, max_new_tokens=50)
print(response2)
print(metrics2)